In [1]:
!pip install espn_api


[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [41]:
from espn_api.football import League
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

In [42]:
LEAGUE_ID = 86952922
espn_s2 = "AEC20e998honXS4Wi0Z8qzlJdam4%2F%2BaApa7apspnhKR0Npb%2FMsF5DuQsFUcHW%2FhPihQun9U6PGITOi2CkbdfDCkUc8druBVhAwT08Lzrvv8oZli8YAuTi9mIWg7YqtorCNOEKPxHpYswnT3q7b885tRDKBJpLCH0T2h4h1p%2B02SfdlDhjEB2gHqFk1xl6tJRNMBiCkZ8i5RttLW6ER9ZvLTmmAdb5ceZhQ27NEMiMf%2BjWSSvwBdnf2roxwt9baw33BVnnITqYVb8FXsaUwm7%2Bm0m9GLQ%2B66%2BU%2Brg%2BQngXm1ekA%3D%3D"
swid = "{B431504E-F779-4C49-B3E8-28DDF7409957}"

def get_league(year):
    league = League(league_id=LEAGUE_ID, year=year, swid=swid, espn_s2=espn_s2)
    return league
    

In [43]:
def query(query):
    conn = sqlite3.connect('weekly_fantasy_data.db')
    return pd.read_sql_query(query, conn)

In [14]:
league = get_league(2025)

In [22]:
league.recent_activity(size=5)[]

[Activity((Team(Brxggzy Bunch),FA ADDED,Player(Chris Rodriguez Jr.)) (Team(Brxggzy Bunch),DROPPED,Player(Cameron Dicker))),
 Activity((Team(Brxggzy Bunch),FA ADDED,Player(Theo Johnson))),
 Activity((Team(Brxggzy Bunch),DROPPED,Player(Joe Mixon))),
 Activity((Team(Brxggzy Bunch),TRADED,Player(Rico Dowdle)) (Team(Deeeej),TRADED,Player(Bijan Robinson)) (Team(Brxggzy Bunch),TRADED,Player(Ashton Jeanty))),
 Activity((Team(RAWDOGS ®),WAIVER ADDED,Player(Rachaad White)))]

In [29]:
%pip install nfl-data-py


[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import nfl_data_py as nfl

In [79]:
query("select * from z_scores where player_name = 'Ashton Jeanty'")

,id,year,player_name,fantasy_pos,week,weekly_points_ppr,log_ppr,z_week_ppr,fantasy_team,created_at,starter
0,None,2025,Ashton Jeanty,RB,2,7.4,0.869232,0.018914,RAWDOGS ®,None,None
1,None,2025,Ashton Jeanty,RB,1,12.0,1.079181,0.480457,RAWDOGS ®,None,None
2,None,2025,Ashton Jeanty,RB,12,24.8,1.394452,1.173532,Deeeej,None,None
3,None,2025,Ashton Jeanty,RB,11,9.4,0.973128,0.247314,Brxggzy Bunch,None,None
4,None,2025,Ashton Jeanty,RB,10,15.3,1.184691,0.712405,Brxggzy Bunch,None,None
5,None,2025,Ashton Jeanty,RB,9,19.9,1.298853,0.963373,Brxggzy Bunch,None,None
6,None,2025,Ashton Jeanty,RB,7,4.4,0.643453,-0.477428,Brxggzy Bunch,None,None
7,None,2025,Ashton Jeanty,RB,6,16.6,1.220108,0.790264,Brxggzy Bunch,None,None
8,None,2025,Ashton Jeanty,RB,5,15.9,1.201397,0.749130,Brxggzy Bunch,None,None
9,None,2025,Ashton Jeanty,RB,4,33.5,1.525045,1.460622,Brxggzy Bunch,None,None


In [90]:
query("select * from z_scores where player_name = 'Alvin Kamara' and year = 2025")

,id,year,player_name,fantasy_pos,week,weekly_points_ppr,log_ppr,z_week_ppr,fantasy_team,created_at,starter
0,None,2025,Alvin Kamara,RB,2,16.0,1.204120,0.755116,Buck Chair,None,None
1,None,2025,Alvin Kamara,RB,1,13.7,1.136721,0.606948,Buck Chair,None,None
2,None,2025,Alvin Kamara,RB,12,3.5,0.544068,-0.695911,Faid XE,None,None
3,None,2025,Alvin Kamara,RB,10,14.5,1.161368,0.661132,Faid XE,None,None
4,None,2025,Alvin Kamara,RB,9,0.7,-0.154902,-2.232494,Faid XE,None,None
5,None,2025,Alvin Kamara,RB,8,6.5,0.812913,-0.104894,Faid XE,None,None
6,None,2025,Alvin Kamara,RB,7,5.9,0.770852,-0.197360,Faid XE,None,None
7,None,2025,Alvin Kamara,RB,6,12.6,1.100371,0.527038,Faid XE,None,None
8,None,2025,Alvin Kamara,RB,5,9.5,0.977724,0.257417,Faid XE,None,None
9,None,2025,Alvin Kamara,RB,4,11.2,1.049218,0.414587,Faid XE,None,None


In [94]:
query("""
      with trades as (select * from player_trades_2025)
      
      select z_scores.player_name, trades.trade_id, 
      trades.to_team_name, sum(weekly_points_ppr) as total_points_ppr, 
      count(distinct z_scores.week) as n,
      sum(z_week_ppr) as total_z
      from z_scores join trades 
      on trades.player_name = z_scores.player_name AND trades.to_team_name = z_scores.fantasy_team 
      where z_scores.year = 2025 and z_scores.week > trades.week
      group by z_scores.player_name, trades.trade_id, trades.to_team_name order by trade_id
      
      
      """)

,player_name,trade_id,to_team_name,total_points_ppr,n,total_z
0,Jaylen Warren,3_3_5,Deeeej,86.90,7,3.253051
1,Kenneth Walker III,3_3_5,Herb,82.20,8,1.978761
2,Tee Higgins,3_3_5,Herb,118.10,8,3.652404
3,Alvin Kamara,3_6_9,Faid XE,64.40,8,-1.370484
4,Jerry Jeudy,3_6_9,Faid XE,49.20,6,-2.463242
5,Puka Nacua,3_6_9,Buck Chair,136.50,7,5.713603
6,Ashton Jeanty,4_1_7,Brxggzy Bunch,81.50,6,2.985057
7,Cam Skattebo,4_1_7,RAWDOGS ®,73.80,4,3.236039
8,Jaxon Smith-Njigba,4_1_7,Brxggzy Bunch,183.70,7,9.596204
9,Rome Odunze,4_1_7,RAWDOGS ®,63.70,6,-0.222984
